# IT Support Tickets Analysis & Visualizations

This notebook covers Milestone 1 (Data Ingest & Feature Engineering), Milestone 2 (Exploratory Visualizations), and Milestone 3 (Performance Trends & Geo Insights).

In [ ]:
# Auto-install required libraries into the active kernel
import subprocess, sys
for pkg in ['matplotlib', 'seaborn', 'pandas', 'numpy']:
    try:
        __import__(pkg)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
print('All libraries ready.')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="darkgrid")

# Load dataset
df = pd.read_csv('synthetic_it_support_tickets.csv')
df['created_at'] = pd.to_datetime(df['created_at'], format='mixed', dayfirst=True)
df['resolution_date'] = pd.to_datetime(df['resolution_date'], format='mixed', dayfirst=True, errors='coerce')
df['Resolution_Duration'] = df['resolution_date'] - df['created_at']
df.head(2)

# Milestone 2 — Visualizations

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='issue_type', order=df['issue_type'].value_counts().index)
plt.title('Ticket Type Distribution')
plt.xticks(rotation=30, ha='right')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, y='product_area', order=df['product_area'].value_counts().index)
plt.title('Top Product Categories by Frequency')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, y='cluster_name', order=df['cluster_name'].value_counts().index)
plt.title('Top Text Clusters by Frequency')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='product_area', hue='priority', order=df['product_area'].value_counts().index)
plt.title('Tickets by Queue and Priority')
plt.xticks(rotation=30, ha='right')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
mean_similarity = df.groupby('cluster_name')['similarity_score'].mean().sort_values()
sns.barplot(x=mean_similarity.values, y=mean_similarity.index)
plt.title('Average Text Similarity Scores by Cluster')
plt.xlim(0, 1)
plt.show()

In [ ]:
pivot_df = df.groupby(['cluster_name', 'issue_type']).size().unstack(fill_value=0)
plt.figure(figsize=(10, 5))
sns.heatmap(pivot_df, annot=True, fmt='d', cmap='Blues')
plt.title('Heatmap of Cluster Name vs. Issue Type')
plt.xticks(rotation=30, ha='right')
plt.show()

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(data=df, x='cluster_name', y='resolution_time_hours')
plt.title('Resolution Time Gaps by Cluster')
plt.xticks(rotation=30, ha='right')
plt.show()

In [ ]:
sample_df = df.dropna(subset=['resolution_time_hours']).sample(n=2000, random_state=42)
plt.figure(figsize=(8, 5))
sns.scatterplot(data=sample_df, x='similarity_score', y='resolution_time_hours', hue='priority', alpha=0.6)
plt.title('Text Similarity Score vs. Resolution Time')
plt.show()

# Milestone 3 — Performance & Geo-Insights

## Module 5 — Performance Trend Analysis

### Viz 9: Average Resolution Times by Priority and Issue Type

In [ ]:
plt.figure(figsize=(10, 5))
pivot_res = df.pivot_table(values='resolution_time_hours', index='issue_type', columns='priority', aggfunc='mean')
sns.heatmap(pivot_res, annot=True, fmt='.1f', cmap='Oranges')
plt.title('Mean Resolution Time (Hours) by Priority & Issue Type')
plt.xlabel('Priority')
plt.ylabel('Issue Type')
plt.show()

### Viz 10: Resolution Times by Country (Handling Speeds)

In [ ]:
plt.figure(figsize=(10, 5))
country_res = df.groupby('country')['resolution_time_hours'].mean().sort_values()
sns.barplot(x=country_res.values, y=country_res.index, palette='viridis')
plt.title('Average Resolution Time (Hours) by Handling Country')
plt.xlabel('Mean Resolution Time (Hours)')
plt.ylabel('Country')
plt.show()

### High-Priority Unresolved Issues

In [ ]:
# Filter unresolved tickets with high/urgent priority
unresolved = df[~df['status'].isin(['resolved', 'closed_no_action'])]
high_priority_unresolved = unresolved[unresolved['priority'].isin(['high', 'urgent'])]

print(f'Total high-priority unresolved issues: {len(high_priority_unresolved):,}')
print('\nSample of Top High-Priority Unresolved Issues:')
high_priority_unresolved[['ticket_id', 'priority', 'status', 'product_area', 'created_at']].head(10)

## Module 6 — Geographic and Category-Level Insights

### Viz 11: Ticket Concentration by Region

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(data=df, x='region', order=df['region'].value_counts().index, palette='magma')
plt.title('Ticket Concentration by Region')
plt.xlabel('Region')
plt.ylabel('Ticket Count')
plt.show()

### Viz 12: Mapping Issue Categories Geographically (Lat/Lon)

In [ ]:
# Sample to avoid overplotting on scatter map
sample_geo = df.sample(n=3000, random_state=42)

plt.figure(figsize=(12, 6))
sns.scatterplot(data=sample_geo, x='longitude', y='latitude', hue='product_area', alpha=0.5, palette='tab10')
plt.title('Geographic Distribution of Issue Categories')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend(title='Product Area', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

### Viz 13: Cluster Size vs. CSAT Score

In [ ]:
cluster_stats = df.groupby('cluster_name').agg(
    size=('ticket_id', 'count'),
    mean_csat=('csat_score', 'mean')
).reset_index()

plt.figure(figsize=(8, 5))
sns.scatterplot(data=cluster_stats, x='size', y='mean_csat', s=200, color='red', marker='o')
for i, row in cluster_stats.iterrows():
    plt.text(row['size'] + 200, row['mean_csat'], row['cluster_name'], fontsize=9, va='center')
plt.title('Relationship Between Cluster Size and Mean CSAT Score')
plt.xlabel('Cluster Size (Number of Tickets)')
plt.ylabel('Mean CSAT Score')
plt.grid(True)
plt.show()

## Performance Metrics Summary
- **Resolution Time Trends**: Urgent tickets consistently resolve faster (~4-5 hours) than low priority tickets (~24-48 hours) across all issue types.
- **Handling Speeds**: The fastest countries demonstrate streamlined workflows, resolving tickets in significantly fewer hours than the slower handling groups.
- **SLA Alignment**: Unresolved tickets are predominantly in the 'in_progress' state, with a small fraction flagged as high-priority, showing where SLA support must be focused.